# Prep for Suo Developmental

In [1]:
import warnings
import os
import sys
import gc
import warnings

In [2]:
import anndata as ad
import scanpy as sc
import copy
import torch
from pathlib import Path
import networkx as nx
from sklearn.neighbors import kneighbors_graph
import numpy as np
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.stats
import scvi

sys.path.append("/home/icb/kemal.inecik/work/codes/tardis")
import tardis
tardis.config = tardis.config_server

In [3]:
adata_file_path = os.path.join(tardis.config.io_directories["processed"], "dataset_complete_Suo.h5ad")
assert os.path.isfile(adata_file_path), f"File not already exist: `{adata_file_path}`"
adata = ad.read_h5ad(adata_file_path)

In [4]:
# PCA, Harmony is calculated already
adata

AnnData object with n_obs × n_vars = 841922 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes'
    uns: 'rank_genes_groups'
    obsm: 'Unintegrated', 'X_pca', 'harmony'

In [5]:
# scanvi
model_name = "suo_scanvi_v2"
dir_path = os.path.join(
    tardis.config.io_directories["models"],
    model_name
)
model_scanvi = scvi.model.SCANVI.load(dir_path=dir_path, adata=adata)

INFO     File /lustre/groups/ml01/workspace/kemal.inecik/tardis_data/models/suo_scanvi_v2/model.pt already         
         downloaded                                                                                                


In [6]:
adata.obsm["scanvi"] = model_scanvi.get_latent_representation()

In [7]:
# scvi
latent_scvi_path = "/lustre/groups/ml01/workspace/kemal.inecik/tardis_data/_temporary/latent/suo_scvi_15_latent.h5ad"
latent_scvi = ad.read_h5ad(latent_scvi_path)

In [8]:
adata.obsm["scvi"] = latent_scvi.X.copy()
del latent_scvi
gc.collect()

283

In [9]:
# `tardis_01_01` and `tardis_02_02` is relevant

In [10]:
latent_tardis_1_path = "/lustre/groups/ml01/workspace/kemal.inecik/tardis_data/_temporary/latent/suo_v01_01_2_encode_tardis_15_latent.h5ad"
latent_tardis_1 = ad.read_h5ad(latent_tardis_1_path)

In [11]:
adata.obsm["tardis_1"] = latent_tardis_1.X.copy()
del latent_tardis_1
gc.collect()

263

In [12]:
latent_tardis_2_path = "/lustre/groups/ml01/workspace/kemal.inecik/tardis_data/_temporary/latent/suo_v02_02_2_encode_tardis_15_latent.h5ad"
latent_tardis_2 = ad.read_h5ad(latent_tardis_2_path)

In [13]:
adata.obsm["tardis_2"] = latent_tardis_2.X.copy()
del latent_tardis_2
gc.collect()

263

In [14]:
latent_invae_path = "/home/icb/kemal.inecik/lustre_workspace/tardis_data/processed/dataset_complete_Suo_invae.h5ad"
latent_invae = ad.read_h5ad(latent_invae_path)

In [15]:
adata.obsm["invae"] = latent_invae.obsm["inveriant_7"].copy()
del latent_invae
gc.collect()

224

In [18]:
updated_obs = pd.read_csv(
    os.path.join(
        "/lustre/groups/ml01/workspace/kemal.inecik/hdca/data", "metadata", "combined_with_hierarchy", "anno_V1.csv"
    ),
    index_col="Unnamed: 0",
)
new_cols = ["LVL3", "LVL2", "LVL1", "LVL0"]

adata.obs = pd.concat([adata.obs.copy(), updated_obs.copy().loc[adata.obs.index][new_cols]], axis=1)
adata.obs["age"] = adata.obs["age"].astype("str").astype("category")

/tmp/ipykernel_4114723/2077463121.py:1: DtypeWarning: Columns (3,7,8,9,11,17,18,20,22,23,27,28,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  updated_obs = pd.read_csv(


In [19]:
adata

AnnData object with n_obs × n_vars = 841922 × 8192
    obs: 'sample_ID', 'organ', 'age', 'cell_type', 'sex', 'sex_inferred', 'concatenated_integration_covariates', 'integration_donor', 'integration_biological_unit', 'integration_sample_status', 'integration_library_platform_coarse', 'n_genes', '_scvi_batch', '_scvi_labels', 'LVL3', 'LVL2', 'LVL1', 'LVL0'
    uns: 'rank_genes_groups', '_scvi_uuid', '_scvi_manager_uuid'
    obsm: 'Unintegrated', 'X_pca', 'harmony', 'scanvi', 'scvi', 'tardis_1', 'tardis_2', 'invae'

In [20]:
working_dir = "/lustre/groups/ml01/workspace/kemal.inecik/sctram_data"
file_name = "suo_developmental_complete.h5ad"
adata.write_h5ad(os.path.join(working_dir, file_name))